In [1]:
# RetailAssist AI - Data Cleaning & Dataset Preparation


# STEP 1 - INSTALL LIBRARIES

!pip install -q datasets


# STEP 2 - IMPORT LIBRARIES

from datasets import load_from_disk
from datasets import concatenate_datasets

import html
import re
import os

In [2]:
# !rm -rf /content/drive

In [3]:

# STEP 3 - PROJECT PATH

from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM"

RAW_DATA_PATH = f"{PROJECT_PATH}/data/raw"

PROCESSED_PATH = f"{PROJECT_PATH}/data/processed2"

os.makedirs(PROCESSED_PATH, exist_ok=True)


Mounted at /content/drive


In [4]:

# ============================================================

# STEP 4 - LOAD RAW DATASETS

# ============================================================

bitext = load_from_disk(
f"{RAW_DATA_PATH}/bitext_customer_support"
)

dialogues = load_from_disk(
f"{RAW_DATA_PATH}/customer_service_dialogues"
)

ecommerce = load_from_disk(
f"{RAW_DATA_PATH}/ecommerce_support"
)

faq = load_from_disk(
f"{RAW_DATA_PATH}/ecommerce_faq"
)

In [5]:
# ============================================================
# STEP 4A - VERIFY DATASET STRUCTURE
# ============================================================

print("\n==============================")
print("BITEXT DATASET")
print("==============================")

print("Columns:")
print(bitext.column_names)

print("\nSample Record:")
print(bitext[0])


print("\n==============================")
print("CUSTOMER SERVICE DIALOGUES")
print("==============================")

print("Columns:")
print(dialogues.column_names)

print("\nSample Record:")
print(dialogues[0])


print("\n==============================")
print("ECOMMERCE SUPPORT")
print("==============================")

print("Columns:")
print(ecommerce.column_names)

print("\nSample Record:")
print(ecommerce[0])


print("\n==============================")
print("ECOMMERCE FAQ")
print("==============================")

print("Columns:")
print(faq.column_names)

print("\nSample Record:")
print(faq[0])


BITEXT DATASET
Columns:
['flags', 'instruction', 'category', 'intent', 'response']

Sample Record:
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}

CUSTOMER SERVICE DIALOGUES
Columns:
['index', 'id', 'description', 'dialogue']

Sample Record:
{'index': 0, 'id': 1, 'description': 'Price inquiry', 'dialogue': 'Customer: Excuse me, could you tell me the price of the apples per pound? Cashier: Certainly! The price for the apples is $1.99 per pound.'}

ECOMMERCE SUPPORT
Columns:
['issue_area', 'issue_category', 'issue_sub_category', 'issue_category_sub_category', 'customer_sentiment', 'product_category', 'product_sub_category', 'issue_complexity', 'agent_experience_level', 'agent_experi

In [6]:
# ============================================================

# STEP 5 - CLEAN TEXT FUNCTION

# ============================================================

import re
import html

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [7]:
# ============================================================
# STEP 6 - STANDARD FORMAT
# ============================================================

SYSTEM_PROMPT = (
    "You are a helpful retail customer support assistant."
)

# ============================================================
# BITEXT
# ============================================================

def format_bitext(example):

    return {
        "instruction": SYSTEM_PROMPT,

        "input": clean_text(
            str(example.get("instruction", ""))
        ),

        "output": clean_text(
            str(example.get("response", ""))
        )
    }


# ============================================================
# DIALOGUES
# ============================================================

def format_dialogues(example):

    customer = str(
        example.get(
            "Customer",
            example.get("human", "")
        )
    )

    agent = str(
        example.get(
            "Agent",
            example.get("assistant", "")
        )
    )

    return {
        "instruction": SYSTEM_PROMPT,

        "input": clean_text(customer),

        "output": clean_text(agent)
    }


# ============================================================
# ECOMMERCE SUPPORT
# ============================================================

def format_ecommerce(example):

    return {
        "instruction": SYSTEM_PROMPT,

        "input": clean_text(
            str(
                example.get(
                    "instruction",
                    example.get("query", "")
                )
            )
        ),

        "output": clean_text(
            str(
                example.get(
                    "response",
                    example.get("answer", "")
                )
            )
        )
    }


# ============================================================
# FAQ
# ============================================================

def format_faq(example):

    return {
        "instruction": SYSTEM_PROMPT,

        "input": clean_text(
            str(
                example.get(
                    "question",
                    example.get("input", "")
                )
            )
        ),

        "output": clean_text(
            str(
                example.get(
                    "answer",
                    example.get("output", "")
                )
            )
        )
    }

In [8]:
# ============================================================

# STEP 7 - CONVERT TO SAME FORMAT

# ============================================================

bitext = bitext.map(
format_bitext,
remove_columns=bitext.column_names
)

dialogues = dialogues.map(
format_dialogues,
remove_columns=dialogues.column_names
)

ecommerce = ecommerce.map(
format_ecommerce,
remove_columns=ecommerce.column_names
)

faq = faq.map(
format_faq,
remove_columns=faq.column_names
)

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/79 [00:00<?, ? examples/s]

In [9]:
# Step 8
#Company specific custom dataset generation

# Step 8.1
store_policies = {

    "return_policy":
        "Products can be returned within 7 days of delivery.",

    "refund_policy":
        "Refunds are processed within 5 business days after approval.",

    "shipping_policy":
        "Orders are usually delivered within 3 to 7 business days.",

    "tracking_policy":
        "Customers can track orders from the Orders section of their account.",

    "payment_methods":
        "We accept UPI, Credit Cards, Debit Cards and Net Banking.",

    "cancellation_policy":
        "Orders can be cancelled before shipment.",

    "damaged_product":
        "Damaged products can be returned or replaced within 7 days.",

    "exchange_policy":
        "Products can be exchanged within 7 days of delivery.",

    "delivery_delay":
        "Customers will be notified if a shipment is delayed.",

    "warranty_policy":
        "Selected products include a manufacturer warranty.",

    "coupon_policy":
        "Coupons can be applied during checkout if valid.",

    "account_support":
        "Customers can reset their password using the Forgot Password option."
}

# ==========================================================
# STEP 8.2  - FAQ QUESTION TEMPLATES
# ==========================================================

faq_templates = {

    "return_policy": [
        "What is your return policy?",
        "How do I return a product?",
        "Can I return an item after delivery?",
        "How many days do I have to return a product?",
        "Can products be returned?"
    ],

    "refund_policy": [
        "When will I receive my refund?",
        "How long does refund processing take?",
        "How many days for refund?",
        "What is the refund process?",
        "When will my money be returned?"
    ],

    "shipping_policy": [
        "How long does delivery take?",
        "What is the shipping timeline?",
        "When will my order arrive?",
        "How many days for delivery?",
        "What are the shipping times?"
    ],

    "tracking_policy": [
        "How do I track my order?",
        "Can I track my shipment?",
        "Where can I check order status?",
        "How to monitor my package?",
        "How can I see delivery progress?"
    ],

    "payment_methods": [
        "What payment methods are accepted?",
        "Can I pay using UPI?",
        "Do you accept credit cards?",
        "Can I pay using debit card?",
        "What payment options are available?"
    ],

    "cancellation_policy": [
        "Can I cancel my order?",
        "How do I cancel an order?",
        "Can shipped orders be cancelled?",
        "What is the cancellation policy?",
        "When can I cancel my order?"
    ],

    "damaged_product": [
        "What if my product arrives damaged?",
        "How do I return a damaged item?",
        "Can damaged products be replaced?",
        "What should I do if I receive a damaged order?",
        "Is damaged item return allowed?"
    ],

    "exchange_policy": [
        "Can I exchange a product?",
        "How does exchange work?",
        "Can I replace an item?",
        "What is your exchange policy?",
        "How many days for exchange?"
    ],

    "delivery_delay": [
        "What if my order is delayed?",
        "How are delivery delays handled?",
        "What happens if shipment is late?",
        "Will I be informed of delays?",
        "How can I check delayed orders?"
    ],

    "warranty_policy": [
        "Do products have warranty?",
        "What is your warranty policy?",
        "How can I claim warranty?",
        "Are products covered under warranty?",
        "How long is the warranty period?"
    ],

    "coupon_policy": [
        "How do I use a coupon?",
        "Can I apply discount codes?",
        "Where do I enter coupon code?",
        "Why is my coupon not working?",
        "Can multiple coupons be used?"
    ],

    "account_support": [
        "How do I reset my password?",
        "I forgot my password.",
        "How can I recover my account?",
        "How do I login again?",
        "How can I change account password?"
    ]
}

# ==========================================================
# STEP 8.3 - QUESTION VARIATIONS
#
# These prefixes generate hundreds of
# natural language variations.
# ==========================================================

prefixes = [
    "",
    "Please tell me ",
    "Can you explain ",
    "I would like to know ",
    "Could you tell me ",
    "Help me understand ",
    "Can you provide information about ",
    "I have a question about ",
    "Can you help me with ",
    "I need information regarding "
]

# ==========================================================
# STEP 8.4 - CREATE DATASET
# ==========================================================

dataset = []

for topic, questions in faq_templates.items():

    answer = store_policies[topic]

    for question in questions:

        for prefix in prefixes:

            dataset.append({
                "instruction": prefix + question,
                "response": answer
            })

# ==========================================================
# STEP 8.5 - ADD EXTRA VARIATIONS
# ==========================================================

extra_starters = [
    "As a customer, ",
    "Before placing an order, ",
    "After delivery, ",
    "For my recent purchase, ",
    "Regarding my order, "
]

expanded_dataset = []

for row in dataset:

    expanded_dataset.append(row)

    for starter in extra_starters:

        expanded_dataset.append({
            "instruction":
                starter + row["instruction"],

            "response":
                row["response"]
        })

# ==========================================================
# STEP 8.6 - REMOVE DUPLICATES
# ==========================================================

seen = set()
final_dataset = []

for item in expanded_dataset:

    key = (
        item["instruction"],
        item["response"]
    )

    if key not in seen:
        seen.add(key)
        final_dataset.append(item)

In [10]:
#8.7 retail custom FAQ format

from datasets import Dataset

retail_faq = Dataset.from_list(final_dataset)

retail_faq = retail_faq.map(
    lambda x: {
        "instruction": SYSTEM_PROMPT,
        "input": x["instruction"],
        "output": x["response"]
    }
)

print(retail_faq[0])

Map:   0%|          | 0/3600 [00:00<?, ? examples/s]

{'instruction': 'You are a helpful retail customer support assistant.', 'response': 'Products can be returned within 7 days of delivery.', 'input': 'What is your return policy?', 'output': 'Products can be returned within 7 days of delivery.'}


In [11]:
# ============================================================

# STEP 8 - MERGE DATASETS

# ============================================================

dataset = concatenate_datasets(
[
bitext,
dialogues,
ecommerce,
faq,
retail_faq
]
)

print("Combined Dataset Size:")
print(len(dataset))

Combined Dataset Size:
31651


In [12]:
PLACEHOLDER_MAP = {

    "{{Order Number}}": [
        "ORD123456",
        "ORD784512",
        "ORD567890",
        "ORD991245",
        "Order #123456"
    ],

    "{{Invoice Number}}": [
        "INV123456",
        "INV987654",
        "Invoice #456789"
    ],

    "{{Tracking Number}}": [
        "TRK123456789",
        "TRK987654321",
        "1Z999AA10123456784",
        "TRACK123456"
    ],

    "{{Account Type}}": [
        "customer account",
        "standard account",
        "premium account",
        "registered account"
    ],

    "{{Account Category}}": [
        "standard",
        "premium",
        "gold",
        "basic"
    ],

    "{{Person Name}}": [
        "John Smith",
        "Sarah Johnson",
        "Michael Brown",
        "Emily Davis",
        "David Wilson"
    ],

    "{{Client First Name}}": [
        "John",
        "Sarah",
        "Michael",
        "Emily",
        "David"
    ],

    "{{Client Last Name}}": [
        "Smith",
        "Johnson",
        "Brown",
        "Davis",
        "Wilson"
    ],

    "{{Salutation}}": [
        "Mr.",
        "Mrs.",
        "Ms.",
        "Dr."
    ],

    "{{Refund Amount}}": [
        "$25",
        "$49.99",
        "$75",
        "$120",
        "$199.99"
    ],

    "{{Money Amount}}": [
        "$20",
        "$50",
        "$100",
        "$150",
        "$250"
    ],

    "{{Currency Symbol}}": [
        "$",
        "₹",
        "€",
        "£"
    ],

    "{{Date Range}}": [
        "3-5 business days",
        "5-7 business days",
        "within 7 days",
        "within 10 business days"
    ],

    "{{Delivery City}}": [
        "New York",
        "Chicago",
        "Dallas",
        "Los Angeles",
        "Seattle"
    ],

    "{{Delivery Country}}": [
        "United States",
        "Canada",
        "United Kingdom",
        "India",
        "Australia"
    ],

    "{{Customer Support Phone Number}}": [
        "+1-800-123-4567",
        "+1-800-555-0199",
        "+1-888-123-4567"
    ],

    "{{Customer Support Email}}": [
        "support@example.com",
        "help@example.com",
        "customerservice@example.com"
    ],

    "{{Website URL}}": [
        "www.examplestore.com",
        "www.shoponline.com",
        "www.myonlinestore.com"
    ],

    "{{Login Page URL}}": [
        "www.examplestore.com/login",
        "www.shoponline.com/login"
    ],

    "{{Store Location}}": [
        "our nearest store",
        "the downtown branch",
        "the local store",
        "the nearest retail outlet"
    ],

    "{{Order Status}}": [
        "currently being processed",
        "shipped",
        "out for delivery",
        "delivered",
        "awaiting shipment"
    ],

    "{{Forgot Password}}": [
        "password reset option",
        "Forgot Password page",
        "password recovery page"
    ],

    "{{Account Recovery}}": [
        "account recovery process",
        "account verification process",
        "identity verification process"
    ],

    "{{Purchase History}}": [
        "order history",
        "purchase history",
        "recent orders section"
    ],

    "{{Live Chat Support}}": [
        "live chat support",
        "online support chat",
        "customer chat service"
    ],
    "{{Online Order Interaction}}": [
        "your recent online order",
        "your purchase",
        "your order request"
    ],

    "{{Customer Support Hours}}": [
        "Monday to Friday, 9 AM to 6 PM",
        "24/7",
        "business hours"
    ],

    "{{Online Company Portal Info}}": [
        "customer portal",
        "online account portal"
    ],

    "{{Settings}}": [
        "account settings",
        "profile settings"
    ],

    "{{Profile}}": [
        "customer profile",
        "user profile"
    ],

    "{{Account Change}}": [
        "account update",
        "account modification"
    ],

    "{{Upgrade Account}}": [
        "upgrade your account",
        "switch to a premium plan"
    ],

    "{{Profile Type}}": [
        "customer profile",
        "premium profile"
    ],

    "{{Order Tracking}}": [
        "order tracking page",
        "tracking section"
    ],

    "{{Track Order}}": [
        "track your order",
        "order tracking page"
    ],

    "{{Forgot PIN}}": [
        "PIN reset page"
    ],

    "{{Client Name}}": [
        "John Smith",
        "Sarah Johnson"
    ],

    "{{Company Name}}": [
        "Example Store"
    ],

    "{{Company}}": [
        "Example Store"
    ],

    "{{Security}}": [
        "security settings",
        "account security"
    ],

    "{{Profile Settings}}": [
        "profile settings"
    ],

    "{{Cancel Purchase}}": [
        "cancel your order"
    ],

    "{{Forgot Key}}": [
        "credential recovery"
    ],

    "{{Account Recovery Page}}": [
        "account recovery page"
    ],

    "{{Reset PIN}}": [
        "reset your PIN"
    ],

    "{{Company Account}}": [
        "business account"
    ],

    "{{PIN Code}}": [
        "1234"
    ],

    "{{Reset Password}}": [
        "password reset page"
    ],

    "{{Feature 1}}": [
        "fast delivery"
    ],

    "{{Feature 2}}": [
        "easy returns"
    ],

    "{{Feature 3}}": [
        "secure payments"
    ],

    "{{Feedback Email Address}}": [
        "feedback@example.com"
    ],

    "{{Client Full Name}}": [
        "John Smith",
        "Sarah Johnson"
    ],

    "{{PIN Settings}}": [
        "PIN settings"
    ],

    "{{PIN Recovery}}": [
        "PIN recovery process"
    ],

    "{{Password Reset Page}}": [
        "password reset page"
    ],

    "{{Reset Access Key}}": [
        "access key reset"
    ],

    "{{Reset Key}}": [
        "credential reset"
    ],

    "{{Customer Service Email}}": [
        "support@example.com"
    ],

    "{{Privacy}}": [
        "privacy settings"
    ],

    "{{Password}}": [
        "your password"
    ],

    "{{Password Recovery Page URL}}": [
        "www.examplestore.com/password-reset"
    ],

    "{{Refund Processing Time}}": [
        "5 business days"
    ],

    "{{PIN Retrieval}}": [
        "PIN recovery process"
    ],

    "{{PIN Management}}": [
        "PIN management settings"
    ],

    "{{Submit}}": [
        "submit"
    ],

    "{{Profile Recovery Page URL}}": [
        "www.examplestore.com/profile-recovery"
    ],

    "{{Forgot Pin Code}}": [
        "PIN recovery page"
    ],

    "{{Online Customer Support Channel}}": [
        "live chat",
        "customer support portal"
    ],

    "{{Forgot Profile Key}}": [
        "profile recovery process"
    ],

    "{{Contact Method}}": [
        "email",
        "phone",
        "live chat"
    ],

    "{{Forgot Account Key}}": [
        "account recovery process"
    ]
}

In [13]:
import re

def remaining_placeholders():

  remaining = 0

  for row in dataset:

      text = (
          row["input"] +
          " " +
          row["output"]
      )

      if re.search(r"\{\{.*?\}\}", text):
          remaining += 1

  print(
      f"Records still containing placeholders: {remaining}"
  )

In [14]:
remaining_placeholders();

Records still containing placeholders: 13041


In [15]:
from collections import Counter
import re

remaining = Counter()

for row in dataset:

    text = (
        str(row["input"]) + " " +
        str(row["output"])
    )

    matches = re.findall(
        r"\{\{.*?\}\}",
        text
    )

    remaining.update(matches)

print("Remaining placeholders:")
for k, v in remaining.most_common(20):
    print(f"{k}: {v}")

Remaining placeholders:
{{Order Number}}: 8029
{{Account Type}}: 5440
{{Account Category}}: 3900
{{Online Order Interaction}}: 2699
{{Customer Support Phone Number}}: 2635
{{Website URL}}: 2534
{{Customer Support Hours}}: 2325
{{Invoice Number}}: 1521
{{Person Name}}: 1249
{{Refund Amount}}: 1187
{{Online Company Portal Info}}: 1055
{{Tracking Number}}: 896
{{Date Range}}: 894
{{Delivery City}}: 840
{{Currency Symbol}}: 745
{{Client Last Name}}: 712
{{Salutation}}: 706
{{Delivery Country}}: 681
{{Settings}}: 625
{{Profile}}: 490


In [16]:
import random
import re

GENERIC_VALUES = {
    "URL": "https://www.example.com",
    "EMAIL": "support@example.com",
    "PHONE": "1-800-555-1234",
    "NUMBER": "123456",
    "DATE": "January 15, 2026"
}


def replace_unknown(match):

    ph = match.group(0).lower()

    if "url" in ph:
        return GENERIC_VALUES["URL"]

    elif "email" in ph:
        return GENERIC_VALUES["EMAIL"]

    elif any(x in ph for x in [
        "phone", "hotline", "contact number",
        "support number", "toll-free"
    ]):
        return GENERIC_VALUES["PHONE"]

    elif "date" in ph:
        return GENERIC_VALUES["DATE"]

    elif any(x in ph for x in [
        "number", "id", "tracking",
        "account", "invoice", "order",
        "reference", "ticket", "pin",
        "zip code"
    ]):
        return GENERIC_VALUES["NUMBER"]

    return "the requested information"


def replace_placeholders(text):

    if not isinstance(text, str):
        return text

    # Step 1: Replace known placeholders
    for pattern, values in compiled_patterns.items():

        text = pattern.sub(
            lambda m: random.choice(values),
            text
        )

    # Step 2: Replace any remaining placeholders
    text = re.sub(
        r"\{\{.*?\}\}",
        replace_unknown,
        text
    )

    return text

In [18]:


def replace_dataset_placeholders(example):

    return {
        "instruction": replace_placeholders(
            example.get("instruction", "")
        ),
        "input": replace_placeholders(
            example.get("input", "")
        ),
        "output": replace_placeholders(
            example.get("output", "")
        )
    }


dataset = dataset.map(
    replace_dataset_placeholders,
    desc="Replacing placeholders"
)

Replacing placeholders:   0%|          | 0/31651 [00:00<?, ? examples/s]

NameError: name 'compiled_patterns' is not defined

In [19]:
remaining_placeholders();

Records still containing placeholders: 13041


In [20]:
# ============================================================
# STEP 9 - REMOVE EMPTY ROWS
# ============================================================

dataset = dataset.filter(
    lambda x:
        len(x["input"].strip()) > 0
        and
        len(x["output"].strip()) > 0
)

print("After Empty Removal:")
print(len(dataset))

Filter:   0%|          | 0/31651 [00:00<?, ? examples/s]

After Empty Removal:
30551


In [21]:
# ============================================================
# STEP 10 - REMOVE DUPLICATES
# ============================================================

seen = set()
keep = []

for i, row in enumerate(dataset):

    input_text = str(row["input"]).lower().strip()
    output_text = str(row["output"]).lower().strip()

    key = (input_text, output_text)

    if key not in seen:
        seen.add(key)
        keep.append(i)

dataset = dataset.select(keep)

print("After Duplicate Removal:")
print(len(dataset))

After Duplicate Removal:
30551


In [22]:
# ============================================================

# STEP 11 - SHUFFLE

# ============================================================

dataset = dataset.shuffle(seed=42)

In [23]:
# ============================================================

# STEP 12 - TRAIN / VALIDATION SPLIT

# ============================================================

split = dataset.train_test_split(
test_size=0.10,
seed=42
)

train_dataset = split["train"]

validation_dataset = split["test"]

print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))

Train: 27495
Validation: 3056


In [24]:
# ============================================================

# STEP 13 - SAVE DATASETS

# ============================================================

train_dataset.save_to_disk(
f"{PROCESSED_PATH}/train"
)

validation_dataset.save_to_disk(
f"{PROCESSED_PATH}/val"
)

print("=================================")
print("Processed dataset saved.")
print("=================================")

print("\nSample Record:")
print(train_dataset[0])

Saving the dataset (0/1 shards):   0%|          | 0/27495 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3056 [00:00<?, ? examples/s]

Processed dataset saved.

Sample Record:
{'instruction': 'You are a helpful retail customer support assistant.', 'input': 'help to use the premium profile', 'output': "I'm on it! I'm delighted to provide guidance on how to make the most of your {{Account Category}} profile: 1. Begin by signing in to our platform using your credentials. 2. Once you're logged in, navigate to your '{{Settings}}' or '{{Profile}}' section. 3. Look for options related to '{{Profile Type}}' or '{{Upgrade Account}}'. 4. Select the suitable option that signifies your desire to switch to the {{Account Category}} profile. 5. Follow the on-screen instructions to complete the upgrade process. If you encounter any difficulties or have additional questions, please don't hesitate to reach out. I'm here to ensure a smooth and enjoyable experience as you unlock the benefits of our {{Account Category}} profile.", 'response': None}
